### **QLoRA Fine-Tuning of Llama 3.2 3B for Text-to-SQL**


In [8]:
#load library
from unsloth import FastLanguageModel, is_bfloat16_supported
import gc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_dataset, Dataset
from trl import SFTTrainer,SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [9]:
import re, sqlite3, time

In [10]:
# Configuration 
MODEL_NAME = "unsloth/llama-3.2-3b-instruct-bnb-4bit"
MAX_LEN = 512                                 
SEED = 3407
N_TRAIN, N_VAL, N_TEST = 7000, 200, 300       

### **1. Load the model and add the LoRA adapter**

In [11]:
# Load the 4-bit quantized base model and its tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_LEN,
    load_in_4bit = True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)

==((====))==  Unsloth 2026.11.5: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Unsloth 2026.11.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).


### **2. Load, clean and split the data**

Each schema and each question is kept only once, then the data is shuffled and cut into train / validation / test. Because every row has a unique schema and a unique question, nothing can leak between the splits.

In [12]:
#load dataset
ds = load_dataset("b-mc2/sql-create-context", split="train")
print(ds)
# convert dataset to a pandas DataFrame
df = ds.to_pandas()[["question", "context", "answer"]].copy()

README.md: 0.00B [00:00, ?B/s]

sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Dataset({
    features: ['answer', 'question', 'context'],
    num_rows: 78577
})


In [13]:
# Cleaning: remove missing values and empty strings
df = df.dropna()
for c in df.columns:
    df[c] = df[c].str.strip()
df = df[(df["question"] != "") & (df["context"] != "") & (df["answer"] != "")]

In [14]:
# keep each schema and each question only once
df = df.drop_duplicates(subset="context").drop_duplicates(subset="question")

# Shuffle then cut into the three splits
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(len(df), "unique examples")
test_df  = df.iloc[:N_TEST]
val_df   = df.iloc[N_TEST : N_TEST + N_VAL]
train_df = df.iloc[N_TEST + N_VAL : N_TEST + N_VAL + N_TRAIN]
print("train / val / test:", len(train_df), len(val_df), len(test_df))

72761 unique examples
train / val / test: 7000 200 300


In [15]:
# leakage checks: no schema and no question shared between two splits
splits = {"train": train_df, "val": val_df, "test": test_df}
names = list(splits)
for col in ["context", "question"]:
    for i in range(3):
        for j in range(i + 1, 3):
            overlap = set(splits[names[i]][col]) & set(splits[names[j]][col])
            assert len(overlap) == 0, f"{col} shared between {names[i]} and {names[j]}"
print("No schema or question is shared between the splits")


No schema or question is shared between the splits


### **3. Define the prompt and apply it to the data**
text template with three slots: the schema, the question and the SQL answer.

In [16]:
prompt_sql = """### Instruction:
Write the SQL query that answers the question, using the table schema.

### Schema:
{}

### Question:
{}

### SQL:
{}"""

# end of sequence token
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    texts = []
    for q, c, a in zip(examples["question"], examples["context"], examples["answer"]):
        texts.append(prompt_sql.format(c, q, a) + EOS_TOKEN)
    return {"text": texts}

def to_ds(d):
    return Dataset.from_pandas(d.reset_index(drop=True), preserve_index=False).map(
        formatting_prompts_func, batched=True
    )

train_ds, eval_ds, test_ds = to_ds(train_df), to_ds(val_df), to_ds(test_df)

# the full training text
print(train_ds[0]["text"])

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

### Instruction:
Write the SQL query that answers the question, using the table schema.

### Schema:
CREATE TABLE table_name_45 (class VARCHAR, city_of_license VARCHAR)

### Question:
What class is the city of black mountain, north carolina?

### SQL:
SELECT class FROM table_name_45 WHERE city_of_license = "black mountain, north carolina"<|eot_id|>


In [17]:
# Token length statistics
lengths = [len(tokenizer(t)["input_ids"]) for t in tqdm(train_ds["text"])]
print("Mean:", np.mean(lengths), "| 95th percentile:", np.percentile(lengths, 95), "| Max:", max(lengths))



100%|██████████| 7000/7000 [00:01<00:00, 3565.78it/s]

Mean: 82.55671428571429 | 95th percentile: 112.0 | Max: 228


### **4. Train**


In [18]:

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_LEN,
        packing = False, 
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        per_device_eval_batch_size = 1,
        warmup_steps = 10,
        num_train_epochs = 2,
        learning_rate = 1e-4,
        lr_scheduler_type = "cosine",
        weight_decay = 0.01,
        optim = "adamw_8bit",
        
        eval_strategy = "steps",
        eval_steps = 100,
        save_strategy = "steps",
        save_steps = 100,
        save_total_limit = 2,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
        
        logging_steps = 20,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth: Padded batches + per-block compile chosen over padding-free (host bound (padded micro batch ~48 ms of GPU work)). Set UNSLOTH_PADDING_CHOICE=padding_free to override.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/7000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

In [19]:
trainer.train()
print("Best checkpoint:", trainer.state.best_model_checkpoint, "| best validation loss:", trainer.state.best_metric)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,000 | Num Epochs = 2 | Total steps = 1,750
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 48,627,712 of 3,261,377,536 (1.49% trained)


Step,Training Loss,Validation Loss
100,0.863700,0.840054
200,0.807300,0.796513
300,0.816200,0.775420
400,0.760300,0.757271
500,0.743600,0.748161
600,0.748400,0.736115
700,0.750300,0.726593
800,0.706200,0.720349
900,0.659400,0.720640
1000,0.702600,0.716089


Best checkpoint: outputs/checkpoint-1700 | best validation loss: 0.7036430239677429


### **5. Evaluation: base model vs fine-tuned model**


In [20]:
def generate_sql(context, question, max_new_tokens=256):
    # Build the prompt with the schema and the question(sql slot empty)
    prompt = prompt_sql.format(context, question, "")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # no gradients 
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,         
            use_cache=True,
        )

    # Keep only the newly generated tokens and decode them
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [21]:
def extract_sql(text):
    #keep only the first SELECT statement
    m = re.search(r"select\b", text, flags=re.I)
    if not m:
        return text.strip()
    # start from the 'select' position up to the first ';'  or double line break
    return text[m.start():].split(";")[0].split("\n\n")[0].strip()

In [22]:
def is_valid(context, sql):
    # Check that the query runs against the schema using explain 
    con = sqlite3.connect(":memory:")
    # returns True if valid and False if it causes an error
    try:
        con.executescript(context)
        con.execute("EXPLAIN " + sql)
        return True
    except Exception:
        return False
    finally:
        con.close()

In [23]:
# Prepare the model for text generation
FastLanguageModel.for_inference(model)

#test dataset
test_eval = test_df.reset_index(drop=True)      
contexts = test_eval["context"].tolist()
questions = test_eval["question"].tolist()
refs = test_eval["answer"].tolist()

In [24]:
# geneartion using Fine-tuned model
t0 = time.time()
preds_ft = [generate_sql(c, q) for c, q in tqdm(zip(contexts, questions), total=len(test_eval))]
time_ft = time.time() - t0

# generation using Base model disabling qlora adapter
t0 = time.time()
with model.disable_adapter():
    preds_base = [generate_sql(c, q) for c, q in tqdm(zip(contexts, questions), total=len(test_eval))]
time_base = time.time() - t0


100%|██████████| 300/300 [22:58<00:00,  4.60s/it]


In [25]:

# the reference queries should almost always be valid with this SQLite check
ref_valid = np.mean([is_valid(c, r) for c, r in zip(contexts, refs)])
print(f"Valid reference queries: {ref_valid * 100:.1f}%")

rows = []
for name, preds, elapsed in [("Base", preds_base, time_base), ("Fine-tuned", preds_ft, time_ft)]:
    rows.append({
        "Model": name,
        "Valid query (%)": round(100 * np.mean([is_valid(c, extract_sql(p)) for c, p in zip(contexts, preds)]), 1),
        "Generation time (min)": round(elapsed / 60, 1),
    })
results = pd.DataFrame(rows)
print(results.to_string(index=False))




Valid reference queries: 99.0%
     Model  Valid query (%)  Generation time (min)
      Base             91.7                   23.0
Fine-tuned             99.3                    3.6
